## Chronos

In [113]:
from chronos import BaseChronosPipeline

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-tiny",
    device_map="cpu",
)

### Model

In [114]:
model = pipeline.model

print(type(model))

<class 'chronos.chronos_bolt.ChronosBoltModelForForecasting'>


In [115]:
pre_transformer_modules = []
encoder_modules = []
decoder_modules = []
post_transformer_modules = []

modules = []
for name, module in model.named_modules():
    modules.append((name, module))



In [116]:
for name, module in modules:
    if "encoder" not in name:
        pre_transformer_modules.append((name, module))
        modules.remove((name, module))
    else:
        break

i = 0
for name, module in pre_transformer_modules:
    if i == 0:
        print(f"Pre-transformer modules: {type(module)}")
    else:
        print(f"{i}: {name}: {type(module)}")
    i += 1

Pre-transformer modules: <class 'chronos.chronos_bolt.ChronosBoltModelForForecasting'>
1: input_patch_embedding: <class 'chronos.chronos_bolt.ResidualBlock'>
2: input_patch_embedding.hidden_layer: <class 'torch.nn.modules.linear.Linear'>
3: input_patch_embedding.output_layer: <class 'torch.nn.modules.linear.Linear'>
4: patch: <class 'chronos.chronos_bolt.Patch'>


In [117]:
for name, module in modules:
    if "encoder" in name and "decoder" not in name:
        encoder_modules.append((name, module))
        modules.remove((name, module))
        
i = 0
for name, module in encoder_modules:
    if i == 0:
        print(f"Encoder modules: {type(module)}")
    else:
        print(f"{i}: {name}: {type(module)}")
    i += 1

Encoder modules: <class 'transformers.models.t5.modeling_t5.T5Stack'>
1: encoder.block.0: <class 'transformers.models.t5.modeling_t5.T5Block'>
2: encoder.block.0.layer.0: <class 'transformers.models.t5.modeling_t5.T5LayerSelfAttention'>
3: encoder.block.0.layer.0.SelfAttention.q: <class 'torch.nn.modules.linear.Linear'>
4: encoder.block.0.layer.0.SelfAttention.v: <class 'torch.nn.modules.linear.Linear'>
5: encoder.block.0.layer.0.SelfAttention.relative_attention_bias: <class 'torch.nn.modules.sparse.Embedding'>
6: encoder.block.0.layer.0.dropout: <class 'torch.nn.modules.dropout.Dropout'>
7: encoder.block.0.layer.1.DenseReluDense: <class 'transformers.models.t5.modeling_t5.T5DenseActDense'>
8: encoder.block.0.layer.1.DenseReluDense.wo: <class 'torch.nn.modules.linear.Linear'>
9: encoder.block.0.layer.1.DenseReluDense.act: <class 'torch.nn.modules.activation.ReLU'>
10: encoder.block.0.layer.1.dropout: <class 'torch.nn.modules.dropout.Dropout'>
11: encoder.block.1.layer: <class 'torch.nn

In [118]:
for name, module in modules:
    if "decoder" in name:
        decoder_modules.append((name, module))
        modules.remove((name, module))

i = 0
for name, module in decoder_modules:
    if i == 0:
        print(f"Decoder modules: {type(module)}")
    else:
        print(f"{i}: {name}: {type(module)}")
    i += 1

Decoder modules: <class 'transformers.models.t5.modeling_t5.T5Stack'>
1: decoder.block.0: <class 'transformers.models.t5.modeling_t5.T5Block'>
2: decoder.block.0.layer.0: <class 'transformers.models.t5.modeling_t5.T5LayerSelfAttention'>
3: decoder.block.0.layer.0.SelfAttention.q: <class 'torch.nn.modules.linear.Linear'>
4: decoder.block.0.layer.0.SelfAttention.v: <class 'torch.nn.modules.linear.Linear'>
5: decoder.block.0.layer.0.SelfAttention.relative_attention_bias: <class 'torch.nn.modules.sparse.Embedding'>
6: decoder.block.0.layer.0.dropout: <class 'torch.nn.modules.dropout.Dropout'>
7: decoder.block.0.layer.1.EncDecAttention: <class 'transformers.models.t5.modeling_t5.T5Attention'>
8: decoder.block.0.layer.1.EncDecAttention.k: <class 'torch.nn.modules.linear.Linear'>
9: decoder.block.0.layer.1.EncDecAttention.o: <class 'torch.nn.modules.linear.Linear'>
10: decoder.block.0.layer.1.dropout: <class 'torch.nn.modules.dropout.Dropout'>
11: decoder.block.0.layer.2.DenseReluDense: <clas

In [119]:
for name, module in modules[20:]:
    if "encoder" not in name and "decoder" not in name:
        post_transformer_modules.append((name, module))

i = 0
for name, module in post_transformer_modules:
    if i == 0:
        print(f"Post-transformer modules: {type(module)}")
    else:
        print(f"{i}: {name}: {type(module)}")
    i += 1

Post-transformer modules: <class 'chronos.chronos_bolt.ResidualBlock'>
1: output_patch_embedding.dropout: <class 'torch.nn.modules.dropout.Dropout'>
2: output_patch_embedding.hidden_layer: <class 'torch.nn.modules.linear.Linear'>
3: output_patch_embedding.act: <class 'torch.nn.modules.activation.ReLU'>
4: output_patch_embedding.output_layer: <class 'torch.nn.modules.linear.Linear'>
5: output_patch_embedding.residual_layer: <class 'torch.nn.modules.linear.Linear'>


### Embeddings

In [120]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from chronos import BaseChronosPipeline
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# LOAD
# ============================================================

pipeline = BaseChronosPipeline.from_pretrained(
    "amazon/chronos-bolt-tiny",
    device_map="cpu",
)

model = pipeline.model


# ============================================================
# STORAGE
# ============================================================

captured = {
    "patches": None,
    "patch_embeddings": None,
    "encoder_inputs": None,
}


# ============================================================
# HOOKS
# ============================================================

def patch_hook(module, inputs, outputs):

    captured["patches"] = outputs.detach().cpu()


def embedding_hook(module, inputs, outputs):

    captured["patch_embeddings"] = outputs.detach().cpu()


def encoder_hook(module, inputs, outputs):

    for x in inputs:

        if torch.is_tensor(x):

            if x.ndim == 3:

                captured["encoder_inputs"] = x.detach().cpu()


# ============================================================
# REGISTER
# ============================================================

h1 = model.patch.register_forward_hook(patch_hook)
# register_forward_hook registers a function to be called every time the module executes a forward pass. The hook function receives the module, its inputs, and its outputs as arguments.
h2 = model.input_patch_embedding.register_forward_hook(
    embedding_hook
)

h3 = model.encoder.register_forward_hook(
    encoder_hook
)
# hook is a handle that can be used to remove the hook later

# ============================================================
# INPUT
# ============================================================

series = torch.tensor([
    11.0, 1.2, 1.4, 1.1,
    0.9, 1.3, 1.5, 1.7,
    1.8, 1.6, 1.4, 1.2,
    1.1, 1.0, 0.8, 0.7,
    0.9, 1.1, 1.3, 1.5,
    1.7, 1.9, 2.0, 1.8,
    1.6, 1.4, 1.2, 1.0,
]).float()


# ============================================================
# RUN THROUGH PIPELINE
# ============================================================

with torch.no_grad():

    forecast = pipeline.predict(
        series,
        prediction_length=8,
    )


# ============================================================
# REMOVE HOOKS
# ============================================================

h1.remove()
h2.remove()
h3.remove()


# ============================================================
# RESULTS
# ============================================================

patches = captured["patches"]
emb = captured["patch_embeddings"]
enc = captured["encoder_inputs"]


print("\n================================================")
print("PATCHES")
print("================================================")

print(patches.shape)

print("\nFirst patch:")
print(patches[0, 0])
print("\nSecond patch:")
print(patches[0, 1])


print("\n================================================")
print("PATCH EMBEDDINGS")
print("================================================")

print(emb.shape)

batch, num_patches, d_model = emb.shape

print("\nNum patches:", num_patches)
print("Embedding dim:", d_model)

print("\nFirst embedding:")
print(emb[0, 0])


print("\n================================================")
print("ENCODER INPUT")
print("================================================")

print(enc.shape)


# ============================================================
# CHECK EQUALITY
# ============================================================

print("\n================================================")
print("ARE PATCH EMBEDDINGS == ENCODER INPUT?")
print("================================================")

diff = torch.abs(emb - enc).mean()

print("Mean absolute difference:", diff.item())


# ============================================================
# NORMS
# ============================================================

norms = torch.norm(emb[0], dim=-1)

plt.figure(figsize=(10, 4))
plt.plot(norms.numpy())
plt.title("Patch Embedding Norms")
plt.xlabel("Patch")
plt.ylabel("Norm")
plt.grid()
plt.show()


# ============================================================
# COSINE SIMILARITY
# ============================================================

X = emb[0].numpy()

sim = cosine_similarity(X)

plt.figure(figsize=(8, 8))
plt.imshow(sim)
plt.colorbar()
plt.title("Patch Embedding Cosine Similarity")
plt.show()


# ============================================================
# PCA
# ============================================================

pca = PCA(n_components=2)

Y = pca.fit_transform(X)

plt.figure(figsize=(8, 6))

for i in range(num_patches):

    plt.scatter(Y[i, 0], Y[i, 1])
    plt.text(Y[i, 0], Y[i, 1], str(i))

plt.title("Patch Embedding PCA")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.grid()
plt.show()


# ============================================================
# TEMPORAL SIMILARITY
# ============================================================

adjacent = []

for i in range(num_patches - 1):

    a = X[i]
    b = X[i + 1]

    cos = np.dot(a, b) / (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )

    adjacent.append(cos)

plt.figure(figsize=(10, 4))
plt.plot(adjacent)
plt.title("Adjacent Patch Similarity")
plt.xlabel("Patch")
plt.ylabel("Cosine similarity")
plt.grid()
plt.show()


# ============================================================
# FORECAST
# ============================================================

print("\n================================================")
print("FORECAST")
print("================================================")

print(forecast)


PATCHES
torch.Size([1, 2, 16])

First patch:
tensor([nan, nan, nan, nan, 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

Second patch:
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

PATCH EMBEDDINGS
torch.Size([1, 2, 256])

Num patches: 2
Embedding dim: 256

First embedding:
tensor([ 1.2612e-01, -2.3305e-03,  9.0789e-02,  8.8850e-02, -1.1941e-02,
         3.1912e-02, -6.4341e-02, -9.4546e-02,  4.3038e-02,  4.9662e-03,
         1.1707e-02, -1.4445e-01,  3.4836e-01,  2.3566e-02, -5.5968e-02,
        -1.2173e-01, -4.6478e-02,  1.3595e-02, -3.5245e-02,  5.6966e-02,
        -1.8253e-01, -8.1013e-03, -4.8491e-02, -6.0737e-03,  5.5036e-02,
        -3.2916e-02,  1.2001e-02, -1.1771e-02, -1.4584e-01, -5.5068e-02,
         1.1478e-01, -7.0098e-02, -2.9848e-02, -1.0509e-02, -8.7233e-02,
         4.1971e-02, -3.2455e-03, -1.0192e-02,  3.4496e-02,  5.5070e-02,
         1.1820e-01,  3.1168e-02,  1.5523e-02,  6.6271e-02,  1.1828e-01,
        -1.4023e-01, -3.2576e-02, -6.

AttributeError: 'NoneType' object has no attribute 'shape'